# Cookbook: Filing Search & Download

Find SEC filings using full-text search and download the actual documents.
This notebook covers the EDGAR Full-Text Search (EFTS) endpoint and the
document download workflow.

**What you'll learn:**
1. Full-text search across all SEC filings
2. Filter by form type, date range, and pagination
3. Inspect search result metadata
4. Download filing documents (HTML, XML)
5. Combine search with company-level filing access

## 1. Setup

In [ ]:
from edgar.client import EdgarClient

edgar_client = EdgarClient(user_agent="Your Name your-email@example.com")

## 2. Basic Full-Text Search

`search()` queries the SEC EDGAR Full-Text Search System (EFTS) — a free
Elasticsearch-powered endpoint that indexes all filing text content.

In [ ]:
results = edgar_client.search("revenue recognition")

print(f"Results returned: {len(results)}\n")
for r in results[:5]:
    print(f"{r.filing_date}  {r.form:<10} {r.company_name}")

In [ ]:
# Each SearchResult auto-renders as an HTML card in Jupyter.
results[0]

## 3. Filter by Form Type

Narrow results to specific filing types like 10-K (annual reports),
10-Q (quarterly), or 8-K (current events).

In [ ]:
# Only 10-K annual reports mentioning "artificial intelligence".
ai_10k = edgar_client.search(
    "artificial intelligence",
    form_types=["10-K"],
    size=10,
)

print(f"10-K filings mentioning 'artificial intelligence': {len(ai_10k)}\n")
for r in ai_10k[:5]:
    print(f"{r.filing_date}  {r.company_name:<40} CIK: {r.cik}")

## 4. Filter by Date Range

Use `start_date` and `end_date` (YYYY-MM-DD format) to scope your search.

In [ ]:
# 8-K filings about cybersecurity incidents in 2024.
cyber_8k = edgar_client.search(
    "cybersecurity incident",
    form_types=["8-K"],
    start_date="2024-01-01",
    end_date="2024-12-31",
    size=10,
)

print(f"8-K cybersecurity filings in 2024: {len(cyber_8k)}\n")
for r in cyber_8k[:5]:
    print(f"{r.filing_date}  {r.company_name}")

## 5. Inspect Search Result Properties

Each `SearchResult` has structured metadata extracted from the EFTS response.

In [ ]:
result = ai_10k[0] if ai_10k else results[0]

print(f"Company:     {result.company_name}")
print(f"CIK:         {result.cik}")
print(f"Form:        {result.form}")
print(f"Filing Date: {result.filing_date}")
print(f"Accession:   {result.accession_number}")
print(f"File Type:   {result.file_type}")
print(f"Description: {result.file_description}")
print(f"Period:      {result.period_ending}")
print(f"URL:         {result.url}")

## 6. Pagination

EFTS returns up to 100 results per request. Use `start` and `size` to paginate.

In [ ]:
# First page of 5 results.
page_1 = edgar_client.search("climate risk", form_types=["10-K"], start=0, size=5)
print(f"Page 1: {len(page_1)} results")
for r in page_1:
    print(f"  {r.filing_date}  {r.company_name}")

# Second page of 5 results.
page_2 = edgar_client.search("climate risk", form_types=["10-K"], start=5, size=5)
print(f"\nPage 2: {len(page_2)} results")
for r in page_2:
    print(f"  {r.filing_date}  {r.company_name}")

## 7. Export Search Results to DataFrame

In [ ]:
from edgar.models import to_dataframe

search_df = to_dataframe(ai_10k)
search_df[["company_name", "form", "filing_date", "cik"]].head(10)

## 8. Download Filing Documents

Once you have a filing URL, use `download()` to fetch the actual document.
SEC filings are typically HTML, XML, or plain text.

In [ ]:
# Get Apple's most recent 10-K filing.
apple = edgar_client.company("AAPL")
filings = apple.get_filings(form="10-K")
latest_10k = filings[0]

print(f"Filing: {latest_10k.form_type} filed {latest_10k.filing_date}")
print(f"URL:    {latest_10k.url}")

In [ ]:
# Download the filing document content.
content = edgar_client.download(latest_10k.url)

# Show the first 500 characters.
print(f"Content type: {type(content).__name__}")
print(f"Content length: {len(content):,} characters\n")
print(content[:500])

## 9. Save to File

Pass a `path` argument to save the downloaded document directly to disk.

In [ ]:
# Uncomment to save to disk:
# edgar_client.download(latest_10k.url, path="apple_10k.html")
# print("Saved to apple_10k.html")

# For this demo, we just show the approach:
print("To save a filing:")
print('  edgar_client.download(url, path="apple_10k.html")')

## 10. Company-Level Filing Browse

For targeted research, browse filings through the Company API rather than search.

In [ ]:
# All recent Apple filings (any form type).
all_filings = apple.get_filings()
print(f"Total filings: {len(all_filings)}\n")

# Group by form type.
form_counts = {}
for f in all_filings:
    form_counts[f.form_type] = form_counts.get(f.form_type, 0) + 1

for form, count in sorted(form_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {form:<12} {count:>3} filings")

## Summary

| Task | Method | Returns |
|------|--------|---------|
| Full-text search | `client.search("query")` | `list[SearchResult]` |
| Filter by form | `search(..., form_types=["10-K"])` | `list[SearchResult]` |
| Filter by date | `search(..., start_date="2024-01-01")` | `list[SearchResult]` |
| Paginate | `search(..., start=100, size=50)` | `list[SearchResult]` |
| Download document | `client.download(url)` | `str` or `bytes` |
| Save to file | `client.download(url, path="file.html")` | `str` or `bytes` |
| Company filings | `company.get_filings(form="10-K")` | `list[Filing]` |